# SwiftLog Logistics Data Science Project

This notebook documents the complete Week 5 Data Science workflow for the SwiftLog Logistics project. It covers data ingestion, data quality checks, cleaning, exploratory analysis, statistical analysis, feature engineering, and business recommendations for operational decision-making.

## 1. Project goal

The objective is to transform raw logistics and operational data into actionable knowledge that can support supplier performance analysis, delivery monitoring, inventory planning, customer value analysis, and predictive analytics for late deliveries or returns.

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

repo_root = Path('/Users/abybatou/Desktop/swiftlof week5')
repo_root


## 2. Load the datasets

In [ ]:
data_files = {
    'customers': repo_root / 'SwiftLog_Customers.csv',
    'orders': repo_root / 'SwiftLog_Orders.csv',
    'order_items': repo_root / 'SwiftLog_Order_Items.csv',
    'products': repo_root / 'SwiftLog_Products.csv',
    'shipments': repo_root / 'SwiftLog_Shipments.csv',
    'delivery_performance': repo_root / 'SwiftLog_Delivery_Performance.csv',
    'inventory': repo_root / 'SwiftLog_Inventory.csv',
    'warehouses': repo_root / 'SwiftLog_Warehouses.csv',
    'employees': repo_root / 'SwiftLog_Employees.csv',
    'returns': repo_root / 'SwiftLog_Returns.csv',
    'suppliers': repo_root / 'SwiftLog_Suppliers.csv',
}

raw = {name: pd.read_csv(path) for name, path in data_files.items()}
raw.keys()


In [ ]:
for name, df in raw.items():
    print(f'\n{name}: shape={df.shape}')
    print(df.head(3).to_string(index=False))


## 3. Data profiling and quality checks

In [ ]:
profile = []
for name, df in raw.items():
    profile.append({
        'Dataset': name,
        'Rows': int(df.shape[0]),
        'Columns': int(df.shape[1]),
        'MissingValues': int(df.isna().sum().sum()),
        'DuplicateRows': int(df.duplicated().sum()),
    })

profile_df = pd.DataFrame(profile)
profile_df


In [ ]:
for name, df in raw.items():
    print(f'\n--- {name} ---')
    print(df.dtypes.to_string())
    print('Missing values by column:')
    print(df.isna().sum().to_dict())


## 4. Data cleaning and transformation

The input data is structurally clean, but we still standardise the date fields, convert numeric columns, and prepare a single analytical data model.

In [ ]:
customers = raw['customers'].copy()
orders = raw['orders'].copy()
order_items = raw['order_items'].copy()
products = raw['products'].copy()
shipments = raw['shipments'].copy()
delivery = raw['delivery_performance'].copy()
inventory = raw['inventory'].copy()
warehouses = raw['warehouses'].copy()
employees = raw['employees'].copy()
returns = raw['returns'].copy()

# conversion to usable types
orders['OrderDate'] = pd.to_datetime(orders['OrderDate'], errors='coerce')
shipments['DelayDays'] = pd.to_numeric(shipments['DelayDays'], errors='coerce')
products['UnitPrice'] = pd.to_numeric(products['UnitPrice'], errors='coerce')
order_items['Quantity'] = pd.to_numeric(order_items['Quantity'], errors='coerce')

# clean strings
for df in [customers, orders, products, shipments, warehouses, employees, returns]:
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].astype(str).str.strip()

print('Order sample after cleaning:')
print(orders.head())


## 5. Merge the operational datasets into a unified orders dataset

In [ ]:
# revenue by order
order_items_with_product = order_items.merge(products[['ProductID', 'Category', 'UnitPrice']], on='ProductID', how='left')
order_items_with_product['Revenue'] = order_items_with_product['Quantity'] * order_items_with_product['UnitPrice']
order_revenue = order_items_with_product.groupby('OrderID', as_index=False)['Revenue'].sum().rename(columns={'Revenue': 'TotalRevenue'})

# merge delivery performance with shipments
shipment_delivery = shipments.merge(delivery, on='ShipmentID', how='left')
shipment_delivery['DelayFlag'] = (shipment_delivery['DelayDays'] > 0).astype(int)
shipment_summary = shipment_delivery.groupby('OrderID', as_index=False).agg(
    WarehouseID=('WarehouseID', 'first'),
    AvgDelayDays=('DelayDays', 'mean'),
    DelayFlag=('DelayFlag', 'max'),
    DeliveryStatus=('Status', lambda s: s.mode().iat[0] if not s.empty else np.nan),
)

# return count per order
return_count = returns.groupby('OrderID', as_index=False).size().rename(columns={'size': 'ReturnCount'})

# create a clean order dataset
orders_enriched = orders.merge(customers, on='CustomerID', how='left')
orders_enriched = orders_enriched.merge(order_revenue, on='OrderID', how='left')
orders_enriched = orders_enriched.merge(shipment_summary, on='OrderID', how='left')
orders_enriched = orders_enriched.merge(return_count, on='OrderID', how='left')
orders_enriched['ReturnCount'] = orders_enriched['ReturnCount'].fillna(0).astype(int)
orders_enriched['HasReturn'] = (orders_enriched['ReturnCount'] > 0).astype(int)
orders_enriched['OrderMonth'] = orders_enriched['OrderDate'].dt.to_period('M').astype(str)

# customer metrics
customer_order_metrics = orders_enriched.groupby('CustomerID', as_index=False).agg(
    TotalCustomerSpend=('TotalRevenue', 'sum'),
    NumberOfOrders=('OrderID', 'count'),
    AverageOrderValue=('TotalRevenue', 'mean')
)
orders_enriched = orders_enriched.merge(customer_order_metrics, on='CustomerID', how='left')

# warehouse metrics
warehouse_inventory = inventory.merge(warehouses, on='WarehouseID', how='left')
warehouse_stock = warehouse_inventory.groupby('WarehouseID', as_index=False)['Stock'].sum().rename(columns={'Stock': 'TotalStock'})
warehouse_employees = employees.groupby('WarehouseID', as_index=False).size().rename(columns={'size': 'EmployeeCount'})
warehouse_summary = warehouse_stock.merge(warehouse_employees, on='WarehouseID', how='left').merge(warehouses, on='WarehouseID', how='left')
warehouse_summary['StockPerEmployee'] = warehouse_summary['TotalStock'] / warehouse_summary['EmployeeCount'].replace(0, np.nan)
orders_enriched = orders_enriched.merge(warehouse_summary[['WarehouseID', 'Location', 'TotalStock', 'EmployeeCount', 'StockPerEmployee']], on='WarehouseID', how='left')

orders_enriched.head()


## 6. Exploratory data analysis (EDA)

In [ ]:
# distribution of order status
plt.figure(figsize=(8, 5))
sns.countplot(data=orders_enriched, x='Status', palette='Set2')
plt.title('Order status distribution')
plt.xlabel('Order status')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

# order trend by month
monthly_orders = orders_enriched.groupby('OrderMonth', as_index=False).size().rename(columns={'size': 'Orders'})
monthly_orders['OrderMonth'] = pd.to_datetime(monthly_orders['OrderMonth'])

plt.figure(figsize=(10, 5))
sns.lineplot(data=monthly_orders, x='OrderMonth', y='Orders', marker='o')
plt.title('Monthly order activity')
plt.xlabel('Month')
plt.ylabel('Orders')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# revenue by customer city
region_sales = orders_enriched.groupby('City', as_index=False)['TotalRevenue'].sum().sort_values('TotalRevenue', ascending=False)
plt.figure(figsize=(10, 5))
sns.barplot(data=region_sales.head(10), x='TotalRevenue', y='City', palette='viridis')
plt.title('Revenue by customer city')
plt.xlabel('Revenue')
plt.ylabel('City')
plt.tight_layout()
plt.show()


In [ ]:
# delivery performance by warehouse location
warehouse_delay = orders_enriched.groupby('Location', as_index=False)['AvgDelayDays'].mean().sort_values('AvgDelayDays', ascending=False)
plt.figure(figsize=(10, 5))
sns.barplot(data=warehouse_delay, x='AvgDelayDays', y='Location', palette='rocket')
plt.title('Average delay by warehouse location')
plt.xlabel('Average delay (days)')
plt.ylabel('Location')
plt.tight_layout()
plt.show()

# revenue by product category
product_summary = order_items_with_product.groupby('Category', as_index=False).agg(
    TotalRevenue=('Revenue', 'sum'),
    TotalUnits=('Quantity', 'sum')
)
plt.figure(figsize=(8, 5))
sns.barplot(data=product_summary, x='Category', y='TotalRevenue', palette='magma')
plt.title('Revenue by product category')
plt.xlabel('Category')
plt.ylabel('Total revenue')
plt.tight_layout()
plt.show()


In [ ]:
# return reasons
return_reason_summary = returns['Reason'].value_counts().reset_index()
return_reason_summary.columns = ['Reason', 'Counts']
plt.figure(figsize=(8, 5))
sns.barplot(data=return_reason_summary, x='Reason', y='Counts', palette='deep')
plt.title('Return reasons')
plt.xlabel('Reason')
plt.ylabel('Counts')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


## 7. Statistical analysis and KPI summary

In [ ]:
# KPI calculations
orders_enriched['DelayFlag'] = orders_enriched['DelayFlag'].fillna(0).astype(int)
orders_enriched['AvgDelayDays'] = orders_enriched['AvgDelayDays'].fillna(0)

summary = {
    'Average delay days': round(float(orders_enriched['AvgDelayDays'].mean()), 2),
    'On-time delivery rate (%)': round((1 - orders_enriched['DelayFlag'].mean()) * 100, 2),
    'Average order revenue': round(float(orders_enriched['TotalRevenue'].fillna(0).mean()), 2),
    'Return rate (%)': round(float(orders_enriched['HasReturn'].mean() * 100), 2),
    'Total stock units': int(orders_enriched['TotalStock'].fillna(0).sum()),
}

summary


In [ ]:
# descriptive statistics
num_cols = ['TotalRevenue', 'AvgDelayDays', 'ReturnCount', 'TotalCustomerSpend', 'AverageOrderValue', 'StockPerEmployee']
orders_enriched[num_cols].describe().round(2)


## 8. Hypothesis testing ideas

The following questions can be tested statistically to support operational decision-making:
- Do delay rates differ across warehouse locations?
- Do return rates vary by product category?
- Do customer spending patterns differ by city or region?
- Are product categories associated with different delivery outcomes?

Recommended methods: ANOVA, chi-square tests, and non-parametric comparisons such as the Kruskal-Wallis test when assumptions are not met.

## 9. Feature engineering

This section creates business features that can be used for predictive modelling and dashboarding.

In [ ]:
feature_df = orders_enriched.copy()
feature_df['LateDeliveryFlag'] = (feature_df['DelayFlag'] == 1).astype(int)
feature_df['RevenuePerOrder'] = feature_df['TotalRevenue'].fillna(0) / feature_df['NumberOfOrders'].replace(0, np.nan)
feature_df['StockToDemandRatio'] = feature_df['TotalStock'].fillna(0) / feature_df['NumberOfOrders'].replace(0, np.nan)
feature_df['CustomerValueBand'] = pd.cut(
    feature_df['TotalCustomerSpend'],
    bins=[-1, 500, 1500, 5000, np.inf],
    labels=['Low', 'Medium', 'High', 'Very High']
)
feature_df['OrderMonthNum'] = pd.to_datetime(feature_df['OrderMonth']).dt.month
feature_df['HasReturnFlag'] = feature_df['HasReturn'].astype(int)

feature_df = feature_df[[
    'OrderID','CustomerID','CustomerName','City','OrderDate','OrderMonth','Status','WarehouseID','Location',
    'TotalRevenue','AvgDelayDays','DelayFlag','LateDeliveryFlag','ReturnCount','HasReturnFlag',
    'TotalCustomerSpend','NumberOfOrders','AverageOrderValue','TotalStock','EmployeeCount','StockPerEmployee',
    'RevenuePerOrder','StockToDemandRatio','CustomerValueBand','OrderMonthNum'
]].copy()

feature_df.head()


## 10. Save the cleaned feature dataset for GitHub

In [ ]:
output_dir = repo_root / 'outputs'
output_dir.mkdir(exist_ok=True)
feature_df.to_csv(output_dir / 'swiftlog_feature_dataset.csv', index=False)
print(f'Feature dataset saved at {output_dir / "swiftlog_feature_dataset.csv"}')


## 11. Business insights

- Order outcomes are spread across delivered, pending, and cancelled statuses, which means operations need proactive monitoring.
- Average delay across shipments is significant and indicates service issues in certain locations.
- Certain regions and warehouses show stronger delay patterns and should be reviewed by operations management.
- Return reasons such as late delivery, wrong item, and damaged goods point to a mixture of delivery and product-quality issues.
- The output feature dataset is suitable for predictive tasks such as late-delivery prediction, return-risk modelling, and customer value analysis.

## 12. Recommendations

1. Prioritise warehouse-by-location performance reviews.
2. Investigate supplier reliability and late-delivery root causes.
3. Reduce return rates through quality controls and packaging improvements.
4. Use customer value features to segment high-value customers.
5. Deploy a predictive model for shipment delay risk and return risk using the engineered feature set.